In [ ]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

# Etapa 4 · Respuestas fundamentadas con un presupuesto verificado

**Para qué.** La recuperación fija el techo; la generación decide cuánto de ese techo se alcanza.
Esta etapa mide esa segunda superficie y construye el ensamblador de prompts que reusan las Etapas 5 y 6.

**Entradas:** la corrida de la Etapa 3 (`results/stage3_<hash>/` + su índice), la evaluación congelada
de la Etapa 2 y `artifacts/full/documents.jsonl`.
**Salidas:** `artifacts/<profile>/results/stage4_<hash>/` (respuestas por ítem y brazo, tablas, 2×2,
comparaciones pareadas, demostración de la guarda, informe de corrida) y la caché del LLM en
`artifacts/<profile>/llm_cache/`.

**Tres filas sobre las mismas preguntas:** libro cerrado · mejor configuración de la Etapa 3 (BM25) · oráculo
(la sección gold). Generador `qwen3:8b`, `think=False`, temperatura 0, schema de Pydantic como `format`,
`num_ctx` explícito en cada llamada. Puntaje sin juez.

**Antes de interpretar:** las salidas smoke solo prueban que el código corre. Dev sirve para desarrollar;
el informe se hace sobre test, una vez fijadas las decisiones. No modifica la evaluación congelada.

## 1. Entorno y decisiones fijadas antes de medir

| Decisión | Valor | Por qué |
|---|---|---|
| `num_ctx` | 8.192 | cómodo en una T4 (§7.2); fijo en todo el experimento (cambiarlo recarga el modelo) |
| `num_predict` | 700 | reserva de salida; `analysis` ≤ 600 y `answer` ≤ 1.200 caracteres entran con holgura |
| Tope del prompt | 8.192 − 700 − 64 = 7.428 | contado con la plantilla de chat; `llm.chat` vuelve a verificarlo |
| Presupuesto de pasajes | 2.500 tokens de Qwen3 | el mismo que la Etapa 3: el "recuperó bien" de la 2×2 es lo que realmente llegó |
| Marcadores | `[doc_id · section_id]` | el modelo cita `section_id`, el nivel del gold |
| Orden del prompt | sistema → pasajes → pregunta | la pregunta queda al final |

Smoke usa `qwen3:1.7b` (§7.4): mismo tokenizer que `qwen3:8b`, así que las cuentas valen igual.

In [ ]:
import environment  # primero: mismas guardas del proyecto (JAX, caché de HF)
from dataclasses import asdict
from pathlib import Path
import csv
import hashlib
import json
import sys
import time
import warnings

SMOKE_TEST = True
EVAL_SPLIT = 'dev'          # 'test' solo cuando las decisiones estén fijadas (y la Etapa 3 corrida en test)
PROFILE = 'smoke' if SMOKE_TEST else 'full'
ART = environment.artifacts_dir(SMOKE_TEST)
DOCUMENTS_PATH = environment.artifacts_dir(False) / 'documents.jsonl'
STAGE3_RESULTS_DIR = None   # opcional: Path('artifacts/full/results/stage3_xxxx') para elegir una corrida
EVAL_DIR = None             # opcional; por defecto eval/ del repo (o la carpeta cuyo FROZEN.json usó la Etapa 3)
MODEL = 'qwen3:1.7b' if SMOKE_TEST else 'qwen3:8b'

RUN_GUARD_DEMO = True       # paso 7: ver saltar la guarda una vez, a propósito (2 llamadas)
RUN_POSITION = False        # paso 6 (opcional): posición del pasaje (3 llamadas por ítem)
POSITION_N_PASSAGES = 12    # pasajes por prompt en la prueba de posición (10-20 sugeridos)
POSITION_BUDGET = 5000      # tokens del bloque de pasajes en esa prueba (igual en las tres posiciones)
REPRO_CHECK_N = 3           # llamadas repetidas sin caché para medir varianza (§7.4)

if EVAL_SPLIT not in ('dev', 'test'):
    raise ValueError('Elegí dev o test explícitamente; no mezclar ambos en una tabla.')
if SMOKE_TEST and EVAL_SPLIT != 'dev':
    raise ValueError('La corrida smoke usa dev; test se reserva para una corrida full.')
if environment.on_colab() and not environment.ARTIFACTS.is_symlink():
    warnings.warn('En Colab, artifacts/ NO está en Drive: no se van a encontrar los resultados de la '
                  'Etapa 3. Poné USE_DRIVE = True en la primera celda y reiniciá el entorno.')
environment.summary()

## 2. Funciones de la etapa

En el notebook, esta celda es una copia de `grounded_stage4.py` (la genera `build_stage4_notebook.py`),
así no depende de haber subido el módulo al repo. Las Etapas 5 y 6 lo importan desde el repo.
El hash del código forma parte de la procedencia de la corrida.

In [ ]:
"""Etapa 4: respuestas fundamentadas con un presupuesto verificado.

El notebook 04_grounded_answers incluye una copia de este módulo, así puede abrirse solo
en Colab (igual que la Etapa 3). Las Etapas 5 y 6 pueden importarlo desde el repo:

    from grounded_stage4 import GroundedAnswer, assemble, build_messages, fit_prompt

Qué contiene:
  - GroundedAnswer: el schema de salida (razonamiento primero, en orden de declaración Y
    alfabético), con el booleano de abstención y las citas.
  - assemble / render_passages / build_messages / fit_prompt: el ensamblador. Saltea un
    pasaje que no entra y sigue con el siguiente (nunca `break`), y verifica el prompt
    completo con la plantilla de chat antes de mandar.
  - score_answer: puntaje sin juez (abstención por booleano, citas contra lo entregado,
    citas contra gold, contención de la clave, veredicto sí/no aparte, flags de
    consistencia). Un error es su propia categoría: nunca abstención, nunca puntuado.
  - two_by_two, summarize_arm, paired_comparison, bootstrap_mean: tablas y cotas de ruido.
  - find_stage3_run / load_stage3_run: lectura de la Etapa 3 verificando su procedencia.

No modifica la evaluación congelada ni los artefactos de la Etapa 3.
"""
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Callable, Literal
import hashlib
import json
import math
import random
import re
import unicodedata

from pydantic import BaseModel, Field


# --- configuración ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Stage4Config:
    """Todo lo que decide qué significa una fila de resultados (va al run_spec)."""
    model: str = "qwen3:8b"
    num_ctx: int = 8192            # fijo en todo el experimento: cambiarlo recarga el modelo
    num_predict: int = 700         # reserva de salida; el prompt tiene que entrar en num_ctx - num_predict
    safety_margin: int = 64        # margen sobre la cuenta propia antes de que llm.chat verifique
    passage_budget: int = 2500     # tokens de Qwen3 para el bloque de pasajes: el mismo de la Etapa 3
    temperature: float = 0.0
    seed: int = 0
    think: bool = False
    bootstrap_samples: int = 2000
    bootstrap_seed: int = 1337

    def __post_init__(self):
        if self.num_predict <= 0 or self.num_ctx <= self.num_predict:
            raise ValueError("num_ctx tiene que ser mayor que num_predict.")
        if self.passage_budget >= self.prompt_limit:
            raise ValueError("El presupuesto de pasajes no deja lugar para sistema y pregunta.")
        if self.temperature != 0.0 or self.think:
            raise ValueError("La etapa fija temperatura 0 y think=False.")

    @property
    def prompt_limit(self) -> int:
        """Tope del prompt completo según NUESTRA cuenta (plantilla incluida)."""
        return self.num_ctx - self.num_predict - self.safety_margin


# --- schema de salida ------------------------------------------------------------------------
class GroundedAnswer(BaseModel):
    # Declarados en el orden en que el modelo debería escribirlos, todos obligatorios, y con
    # nombres cuyo orden alfabético coincide ("analysis" < "answer" < "can_answer" <
    # "citations" < "verdict"): el razonamiento queda primero en cualquier servidor.
    analysis: str = Field(max_length=600, description="razonamiento breve: qué pasaje responde y por qué")
    answer: str = Field(max_length=1200, description="respuesta final, breve, en español")
    can_answer: bool = Field(description="false si el contexto no alcanza para responder")
    citations: list[str] = Field(description="section_id de los pasajes usados, copiados del marcador")
    verdict: Literal["si", "no", "no_aplica"] = Field(description="para preguntas de sí/no; si no, no_aplica")


def check_schema_order(schema: type[BaseModel] = GroundedAnswer) -> list[str]:
    """Falla si el campo de razonamiento no queda primero en declaración y en orden alfabético."""
    props = list(schema.model_json_schema()["properties"])
    required = schema.model_json_schema().get("required", [])
    if props[0] != "analysis" or sorted(props)[0] != "analysis":
        raise AssertionError(f"El razonamiento tiene que ir primero: {props}")
    if set(required) != set(props):
        raise AssertionError(f"Todos los campos deben ser obligatorios: {required} vs {props}")
    return props


# --- prompts ---------------------------------------------------------------------------------
SYSTEM_GROUNDED = """Sos un asistente que responde preguntas sobre un corpus de leyes argentinas.
Respondé usando EXCLUSIVAMENTE los pasajes que se te entregan, delimitados por marcadores
de la forma [doc_id · section_id]. No uses conocimiento propio ni completes con suposiciones.

Devolvé un objeto JSON con estos campos, en este orden:
- analysis: dos o tres oraciones. Indicá qué pasaje contiene la respuesta y qué dice. Si
  ninguno la contiene, decilo.
- answer: la respuesta, breve y directa, en español. Copiá cifras, plazos, fechas y nombres
  tal como aparecen en el pasaje. Si la pregunta es de sí o no, empezá con "Sí" o "No".
  Si los pasajes no alcanzan, escribí exactamente: "No hay información suficiente en los
  pasajes para responder."
- can_answer: true solo si los pasajes contienen la respuesta; false en caso contrario.
- citations: la lista de section_id de los pasajes que usaste, copiados exactamente del
  marcador (por ejemplo, LEY-24240#ART-4). Lista vacía si can_answer es false.
- verdict: "si" o "no" si la pregunta se responde con sí o no; "no_aplica" en otro caso."""

SYSTEM_CLOSED_BOOK = """Sos un asistente que responde preguntas sobre leyes argentinas.
No se te entrega ningún pasaje: respondé solo con lo que sepas. Si no lo sabés con
seguridad, o la pregunta depende de un documento o de datos particulares que no tenés,
no adivines.

Devolvé un objeto JSON con estos campos, en este orden:
- analysis: dos o tres oraciones sobre qué sabés del tema y qué tan seguro estás.
- answer: la respuesta, breve y directa, en español. Si la pregunta es de sí o no, empezá
  con "Sí" o "No". Si no podés responder, escribí exactamente: "No tengo información
  suficiente para responder."
- can_answer: true solo si podés responder; false en caso contrario.
- citations: lista vacía (no tenés pasajes para citar).
- verdict: "si" o "no" si la pregunta se responde con sí o no; "no_aplica" en otro caso."""

SEPARATOR = "\n\n"


def passage_block(p: dict) -> str:
    """Marcador [doc_id · section_id] + el texto con su prefijo (doc_id | título de sección).

    Es el bloque de la Etapa 3 sin el chunk_id: el modelo cita section_id, que es el nivel
    del gold. Varios hijos de una misma sección comparten marcador.
    """
    return f"[{p['doc_id']} · {p['section_id']}]\n{p['index_text']}"


def render_passages(passages: list[dict]) -> str:
    return SEPARATOR.join(passage_block(p) for p in passages)


def user_message(question: str, passages: list[dict] | None) -> str:
    """Pasajes primero, pregunta al final. passages=None -> libro cerrado (sin bloque)."""
    if passages is None:
        return f"PREGUNTA: {question}"
    body = render_passages(passages) if passages else "(no se entregó ningún pasaje)"
    return f"PASAJES:\n{body}\n\nPREGUNTA: {question}"


def build_messages(system: str, question: str, passages: list[dict] | None,
                   history: list[dict] | None = None) -> list[dict]:
    """Sistema + historial (Etapas 5/6) + el turno actual con sus pasajes."""
    return ([{"role": "system", "content": system}] + list(history or [])
            + [{"role": "user", "content": user_message(question, passages)}])


# --- el ensamblador --------------------------------------------------------------------------
def assemble(passages: list[dict], budget: int, count: Callable[[str], int],
             render: Callable[[list[dict]], str] = render_passages) -> tuple[list[dict], list[dict], int]:
    """Llena `budget` tokens con pasajes en el orden dado. Devuelve (entregados, salteados, usados).

    Un pasaje que no entra se saltea y se registra, y se sigue con el siguiente: un `break`
    en el primero que no entra descartaría también los más chicos que vienen después.
    `count` cuenta texto pelado (llm.count_tokens): la suma por bloque se verifica al final
    contra la cuenta del bloque renderizado completo, por si un borde de BPE agrega tokens.
    """
    if budget <= 0:
        raise ValueError("budget tiene que ser positivo")
    sep = count(SEPARATOR)
    delivered, skipped, used = [], [], 0
    for p in passages:
        n = count(passage_block(p)) + (sep if delivered else 0)
        if used + n > budget:
            skipped.append(p)
            continue
        delivered.append(p)
        used += n
    actual = count(render(delivered)) if delivered else 0
    while delivered and actual > budget:   # raro: la suma subestimó por bordes de tokens
        skipped.insert(0, delivered.pop())
        actual = count(render(delivered)) if delivered else 0
    return delivered, skipped, actual


@dataclass
class FittedPrompt:
    messages: list[dict]
    delivered: list[dict]
    skipped_ids: list[str]
    passage_tokens: int                 # cuenta del bloque de pasajes renderizado (texto pelado)
    expected_prompt_tokens: int | None  # cuenta con la plantilla de chat (llm.count_prompt_tokens)
    prompt_limit: int

    @property
    def fits(self) -> bool:
        return self.expected_prompt_tokens is None or self.expected_prompt_tokens <= self.prompt_limit


def fit_prompt(system: str, question: str, candidates: list[dict] | None, cfg: Stage4Config,
               count: Callable[[str], int], count_messages: Callable[[list[dict]], int | None],
               history: list[dict] | None = None, passage_budget: int | None = None) -> FittedPrompt:
    """Ensambla y cuenta el prompt COMPLETO con la plantilla. Si no entra, saca el último pasaje.

    candidates=None es libro cerrado. El presupuesto de pasajes es fijo (comparable entre
    filas); el tope del prompt completo es num_ctx - num_predict - margen.
    """
    budget = cfg.passage_budget if passage_budget is None else passage_budget
    if candidates is None:
        delivered, skipped, used = [], [], 0
    else:
        delivered, skipped, used = assemble(candidates, budget, count)
    skipped_ids = [p.get("chunk_id", p["section_id"]) for p in skipped]
    while True:
        msgs = build_messages(system, question, None if candidates is None else delivered, history)
        expected = count_messages(msgs)
        if expected is None or expected <= cfg.prompt_limit or not delivered:
            break
        removed = delivered.pop()
        skipped_ids.append(removed.get("chunk_id", removed["section_id"]))
        used = count(render_passages(delivered)) if delivered else 0
    return FittedPrompt(msgs, delivered, skipped_ids, used, expected, cfg.prompt_limit)


# --- normalización y puntaje sin juez ----------------------------------------------------------
def normalize_answer(text: str) -> str:
    """La misma normalización de la Etapa 3: minúsculas, sin acentos, sin '(12)', sin
    separadores de miles, sin puntuación. 'Ley 24.240' -> 'ley 24240'; '1,5' -> '1 5'
    (igual en la clave y en la respuesta, así que las comas decimales coinciden)."""
    text = unicodedata.normalize("NFD", (text or "").lower())
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"\(\s*\d+\s*\)", " ", text)
    text = re.sub(r"(?<=\d)[.,\s](?=\d{3}(?:\D|$))", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


def contains_key(text: str, keys: list[str] | None) -> bool:
    normalized = f" {normalize_answer(text)} "
    return any(bool(normalize_answer(k)) and f" {normalize_answer(k)} " in normalized for k in keys or [])


def normalize_citation(c: str) -> str:
    """'[LEY-24240 · LEY-24240#ART-4]', 'LEY-24240 · LEY-24240#ART-4' -> 'LEY-24240#ART-4'."""
    c = (c or "").strip().strip("[]()").strip()
    parts = [x.strip() for x in re.split(r"\s*[·|]\s*", c) if x.strip()]
    with_hash = [x for x in parts if "#" in x]
    return (with_hash[0] if with_hash else (parts[-1] if parts else "")).strip("[]").strip()


# Solo para FLAGS de consistencia (inspección manual): la abstención se puntúa con el booleano.
_ABSTAIN_TEXT = re.compile(
    r"\b(no hay (informacion|datos|suficiente)|informacion (insuficiente|suficiente)|"
    r"no (tengo|dispongo de|cuento con) (informacion|datos)|no (es posible|puedo) (responder|determinar|saber)|"
    r"no (se )?(menciona|especifica|indica|encuentra|figura|aparece)|no lo se|no se puede determinar)\b")


def looks_like_abstention(answer: str) -> bool:
    return bool(_ABSTAIN_TEXT.search(normalize_answer(answer)))


ERROR = "error"
ABSTAINED = "abstained"
ANSWERED = "answered"


def score_answer(item: dict, answer: GroundedAnswer | None, error: str | None,
                 delivered: list[dict], closed_book: bool = False) -> dict:
    """Métricas de una respuesta. `item` es una fila de single_turn.jsonl.

    - error (r.error, ContextOverflow, PARSE_FAILED...): outcome 'error', nada más se puntúa.
    - abstención: SOLO el booleano can_answer.
    - correct: con respuesta -> respondió + clave contenida (+ veredicto si es sí/no);
      sin respuesta (fuera del corpus) -> se abstuvo.
    """
    gold = {(g["doc_id"], g.get("section_id")) for g in item.get("gold") or []}
    gold_sections = {s for _, s in gold if s}
    gold_docs_whole = {d for d, s in gold if s is None}
    keys = item.get("answer_key")
    answerable = bool(item.get("answerable", True))
    delivered_sections = {p["section_id"] for p in delivered}
    m: dict = {
        "answerable": answerable,
        "n_passages": len(delivered),
        # Libro cerrado no recibe pasajes: sus métricas de entrega no aplican (None, no 0).
        "gold_delivered": (any((p["doc_id"], p["section_id"]) in gold or p["doc_id"] in gold_docs_whole
                               for p in delivered) if answerable and not closed_book else None),
        "answer_delivered": (any(contains_key(p.get("text", ""), keys) for p in delivered)
                             if answerable and keys and not closed_book else None),
    }
    if error is not None or answer is None:
        m.update(outcome=ERROR, error=error or "sin respuesta parseada", correct=None)
        return m
    abstained = not answer.can_answer
    cited = [normalize_citation(c) for c in answer.citations]
    cited = [c for c in cited if c]
    delivered_fold = {s.casefold() for s in delivered_sections}
    invalid = [c for c in cited if c.casefold() not in delivered_fold]
    cited_gold = any(c.casefold() in {s.casefold() for s in gold_sections}
                     or c.split("#")[0].casefold() in {d.casefold() for d in gold_docs_whole} for c in cited)
    key_hit = contains_key(answer.answer, keys) if answerable and keys else None
    expected_verdict = item.get("expected_verdict")
    verdict_ok = None
    if expected_verdict:
        verdict_ok = {"yes": "si", "no": "no"}[expected_verdict] == answer.verdict
    if answerable:
        correct = (not abstained) and bool(key_hit) and (verdict_ok is not False)
    else:
        correct = abstained
    text_abstains = looks_like_abstention(answer.answer)
    m.update(
        outcome=ABSTAINED if abstained else ANSWERED,
        error=None,
        can_answer=answer.can_answer,
        correct=correct,
        key_hit=key_hit,
        expected_verdict=expected_verdict,
        verdict=answer.verdict,
        verdict_ok=verdict_ok,
        citations=cited,
        n_citations=len(cited),
        invalid_citations=invalid,
        cited_not_delivered=bool(invalid),
        cited_gold=cited_gold if answerable else None,
        abstained_with_answer_in_context=(abstained and bool(m["answer_delivered"] or m["gold_delivered"])
                                          if answerable and not closed_book else None),
        # Flags (heurísticos, para abrir la caja; no cambian ningún puntaje):
        flag_says_cannot_but_answers=abstained and (bool(key_hit) or not text_abstains),
        flag_says_can_but_abstains=(not abstained) and text_abstains,
        flag_answers_without_citation=(not abstained) and not cited and not closed_book,
    )
    return m


# --- tablas y estadística --------------------------------------------------------------------
def bootstrap_mean(values: list[float], seed: int = 1337, samples: int = 2000):
    if not values:
        return None, None
    rng = random.Random(seed)
    means = sorted(sum(rng.choices(values, k=len(values))) / len(values) for _ in range(samples))
    return means[int(0.025 * (samples - 1))], means[int(0.975 * (samples - 1))]


def _rate(rows: list[dict], key: str, where: Callable[[dict], bool] = lambda r: True):
    vals = [r["metrics"].get(key) for r in rows if where(r) and r["metrics"].get(key) is not None]
    return (sum(bool(v) for v in vals) / len(vals) if vals else None), len(vals)


def summarize_arm(rows: list[dict], cfg: Stage4Config) -> dict:
    """Una fila de la tabla de la etapa. Los errores se cuentan aparte y no se puntúan."""
    scored = [r for r in rows if r["metrics"]["outcome"] != ERROR]
    ans = [r for r in scored if r["metrics"]["answerable"]]
    ooc = [r for r in scored if not r["metrics"]["answerable"]]
    correct = [float(r["metrics"]["correct"]) for r in scored]
    lo, hi = bootstrap_mean(correct, cfg.bootstrap_seed, cfg.bootstrap_samples)
    llm_rows = [r.get("llm") or {} for r in rows]
    deltas = [l["expected_prompt_tokens"] - l["prompt_tokens"] for l in llm_rows
              if l.get("expected_prompt_tokens") is not None and l.get("prompt_tokens") is not None]
    out = {
        "n": len(rows),
        "errors": len(rows) - len(scored),
        "correct": sum(correct) / len(correct) if correct else None,
        "correct_ci95_low": lo, "correct_ci95_high": hi,
        "correct_answerable": _rate(ans, "correct")[0],
        "n_answerable": len(ans),
        "abstention_ooc": _rate(ooc, "correct")[0],
        "n_ooc": len(ooc),
        "abstained_answerable": (sum(r["metrics"]["outcome"] == ABSTAINED for r in ans) / len(ans)) if ans else None,
        "key_hit": _rate(ans, "key_hit")[0],
        "verdict_ok": _rate(ans, "verdict_ok")[0],
        "n_yes_no": _rate(ans, "verdict_ok")[1],
        "cited_not_delivered": _rate(scored, "cited_not_delivered")[0],
        "cited_gold": _rate(ans, "cited_gold")[0],
        "gold_delivered": _rate(rows, "gold_delivered")[0],
        "answer_delivered": _rate(rows, "answer_delivered")[0],
        "abstained_with_answer_in_context": _rate(ans, "abstained_with_answer_in_context")[0],
        "flags": sum(any(r["metrics"].get(f) for f in ("flag_says_cannot_but_answers", "flag_says_can_but_abstains",
                                                         "flag_answers_without_citation")) for r in scored),
        "mean_passage_tokens": _mean([r.get("passage_tokens") for r in rows]),
        "mean_expected_prompt_tokens": _mean([l.get("expected_prompt_tokens") for l in llm_rows]),
        "mean_server_prompt_tokens": _mean([l.get("prompt_tokens") for l in llm_rows]),
        "max_abs_delta_tokens": max((abs(d) for d in deltas), default=None),
        "mean_ms": _mean([l.get("ms") for l in llm_rows if not l.get("cached")]),
    }
    return out


def _mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None


def two_by_two(rows: list[dict], retrieval_key: str = "gold_delivered") -> dict:
    """Por pregunta con respuesta: recuperó bien/mal × respondió bien/mal. Errores aparte."""
    cells = {"ret_ok_ans_ok": [], "ret_ok_ans_bad": [], "ret_bad_ans_ok": [], "ret_bad_ans_bad": [], "error": []}
    for r in rows:
        m = r["metrics"]
        if not m["answerable"]:
            continue
        if m["outcome"] == ERROR:
            cells["error"].append(r["item_id"])
            continue
        ret = "ret_ok" if m.get(retrieval_key) else "ret_bad"
        ans = "ans_ok" if m["correct"] else "ans_bad"
        cells[f"{ret}_{ans}"].append(r["item_id"])
    return {"retrieval_criterion": retrieval_key, "counts": {k: len(v) for k, v in cells.items()}, "items": cells}


def failure_kind(m: dict) -> str:
    """Etiqueta para abrir la caja (no reemplaza mirar la respuesta)."""
    if m["outcome"] == ERROR:
        return "error"
    if m["correct"]:
        return "ok"
    if not m["answerable"]:
        return "respondió una pregunta fuera del corpus"
    if m["outcome"] == ABSTAINED:
        return "se abstuvo con la respuesta en el contexto" if m.get("abstained_with_answer_in_context") else "se abstuvo"
    if m.get("verdict_ok") is False and m.get("key_hit"):
        return "veredicto sí/no equivocado"
    if m.get("gold_delivered") is False and m.get("n_citations") and not m.get("cited_not_delivered"):
        return "fiel al pasaje equivocado (candidato: verificar a mano)"
    if m.get("gold_delivered"):
        return "lo tenía, respondió otra cosa"
    return "respondió sin la clave"


def paired_comparison(rows_a: list[dict], rows_b: list[dict], metric: str = "correct",
                      seed: int = 1337, samples: int = 2000) -> dict:
    """B contra A sobre los mismos ítems: victorias/derrotas/empates, test de signos y bootstrap
    de la diferencia. Un ítem con error en cualquiera de las dos filas se excluye (y se cuenta)."""
    a = {r["item_id"]: r for r in rows_a}
    b = {r["item_id"]: r for r in rows_b}
    if a.keys() != b.keys():
        raise ValueError("La comparación pareada requiere los mismos ítems.")
    wins = losses = ties = excluded = 0
    diffs = []
    for k in sorted(a):
        p, q = a[k]["metrics"].get(metric), b[k]["metrics"].get(metric)
        if p is None or q is None:
            excluded += 1
            continue
        p, q = int(bool(p)), int(bool(q))
        wins += q > p
        losses += q < p
        ties += q == p
        diffs.append(q - p)
    n = wins + losses
    p_value = min(1.0, 2 * sum(math.comb(n, i) for i in range(min(wins, losses) + 1)) / 2 ** n) if n else 1.0
    lo, hi = bootstrap_mean(diffs, seed, samples)
    return {"metric": metric, "wins": wins, "losses": losses, "ties": ties, "excluded_errors": excluded,
            "delta": (sum(diffs) / len(diffs)) if diffs else None, "delta_ci95": [lo, hi],
            "sign_test_p_two_sided": p_value}


def min_discordant_for_significance(alpha: float = 0.05) -> int:
    """Discordantes, todas en el mismo sentido, que hacen falta para p < alpha (p = 2 * 0.5^n)."""
    n = 1
    while 2 * 0.5 ** n >= alpha:
        n += 1
    return n


# --- lectura de la Etapa 3, con procedencia ---------------------------------------------------
def canonical_hash(value) -> str:
    blob = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def read_jsonl(path) -> list[dict]:
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]


def find_stage3_run(art: Path, split: str, profile: str, override: Path | None = None) -> Path:
    """La carpeta results/stage3_<hash>/ terminada para este split y perfil.

    Con varias candidatas toma la más reciente y lo avisa; fijá STAGE3_RESULTS_DIR para elegir.
    """
    if override is not None:
        override = Path(override)
        if not (override / "retrieval.jsonl").is_file():
            raise FileNotFoundError(f"{override} no tiene retrieval.jsonl de la Etapa 3.")
        return override
    found = []
    for spec_path in sorted((Path(art) / "results").glob("stage3_*/run_spec.json")):
        d = spec_path.parent
        spec = json.loads(spec_path.read_text(encoding="utf-8"))
        report = d / "run_report.json"
        status = json.loads(report.read_text(encoding="utf-8")).get("status") if report.is_file() else None
        if spec.get("split") == split and spec.get("profile") == profile and status == "ok" \
                and (d / "retrieval.jsonl").is_file():
            found.append((json.loads(report.read_text(encoding="utf-8")).get("finished_at", ""), d))
    if not found:
        raise FileNotFoundError(
            f"No hay una corrida terminada de la Etapa 3 con split={split!r} y perfil={profile!r} en "
            f"{Path(art) / 'results'}. Corré 03_retrieval con esos valores (SMOKE_TEST y EVAL_SPLIT) primero.")
    found.sort()
    if len(found) > 1:
        print("AVISO: varias corridas de la Etapa 3 coinciden; uso la más reciente. Fijá STAGE3_RESULTS_DIR para elegir:")
        for when, d in found:
            print("   ", when, d)
    return found[-1][1]


def load_stage3_run(run_dir: Path, art: Path) -> dict:
    """run_spec + filas de retrieval + chunks del índice, verificando que sean del mismo índice."""
    run_dir = Path(run_dir)
    spec = json.loads((run_dir / "run_spec.json").read_text(encoding="utf-8"))
    rows = read_jsonl(run_dir / "retrieval.jsonl")
    index_dir = Path(art) / "index" / f"stage3_{spec['index_key'][:16]}"
    manifest_path = index_dir / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta el índice de la Etapa 3 que usó esta corrida: {index_dir}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if canonical_hash(manifest) != spec["index_key"]:
        raise ValueError("El manifiesto del índice no corresponde al index_key de la corrida de la Etapa 3.")
    chunks = read_jsonl(index_dir / "chunks.jsonl")
    if canonical_hash(chunks) != manifest["chunks_hash"]:
        raise ValueError("chunks.jsonl cambió después de construir el índice de la Etapa 3.")
    if {r["run_key"] for r in rows} != {canonical_hash(spec)}:
        raise ValueError("retrieval.jsonl tiene filas de otra corrida que la de run_spec.json.")
    if [r["item_id"] for r in rows] != spec["item_ids"]:
        raise ValueError("retrieval.jsonl no tiene exactamente los ítems de run_spec.json.")
    return {"dir": run_dir, "spec": spec, "rows": {r["item_id"]: r for r in rows},
            "index_dir": index_dir, "manifest": manifest, "chunks": {c["chunk_id"]: c for c in chunks}}


def oracle_passages(item: dict, docs: dict[str, dict], chunks_by_section: dict[str, list[dict]],
                    budget: int, count: Callable[[str], int]) -> tuple[list[dict], str]:
    """La sección gold entera, con el mismo formato de bloque que los hijos.

    Si una sección gold sola no entra en el presupuesto, se entregan sus hijos de la Etapa 3
    en orden de documento (el ensamblador corta). Devuelve (pasajes candidatos, modo).
    """
    out, mode = [], "section"
    for g in item.get("gold") or []:
        doc = docs[g["doc_id"]]
        secs = [s for s in doc["sections"] if s["section_id"] == g["section_id"]] if g.get("section_id") else []
        if not secs:
            raise ValueError(f"{item['item_id']}: gold sin sección utilizable {g}")
        s = secs[0]
        body = doc["text"][s["start"]:s["end"]]
        prefix = f"{doc['doc_id']} | {s['title']}\n"
        unit = {"chunk_id": s["section_id"], "doc_id": doc["doc_id"], "section_id": s["section_id"],
                "text": body, "index_text": prefix + body}
        if count(passage_block(unit)) <= budget:
            out.append(unit)
        else:
            mode = "children_fallback"
            out.extend(sorted(chunks_by_section.get(s["section_id"], []), key=lambda c: c["start"]))
    return out, mode


IMPLEMENTATION_SHA256 = 'e77a43af4c70e4a7caec344ef7a473ce7a2de64d0f9d3403ebea564fe3c3a39c'

In [ ]:
CFG = Stage4Config(model=MODEL)
check_schema_order()
print(json.dumps(asdict(CFG), indent=2), '\nprompt_limit:', CFG.prompt_limit)
print('Schema:', list(GroundedAnswer.model_json_schema()['properties']))

## 3. Insumos: Etapa 3, evaluación congelada y corpus, con su procedencia

Se toma la corrida terminada de la Etapa 3 para este split y perfil, y se verifica que su índice
(manifiesto y `chunks.jsonl`) sea el que usó. La evaluación es la carpeta cuyo `FROZEN.json` coincide
con el de la corrida de la Etapa 3; `schema.load_eval` comprueba los hashes de los tres archivos y del corpus.
Los ítems son exactamente los de la corrida de la Etapa 3, así las tres filas comparten preguntas.

In [ ]:
import schema
import llm
from console import StageRun, print_table, verdict, clip, wrap
from store import ResultsLog, write_json, write_jsonl

STAGE3_DIR = find_stage3_run(ART, EVAL_SPLIT, PROFILE, STAGE3_RESULTS_DIR)
S3 = load_stage3_run(STAGE3_DIR, ART)
print('Etapa 3:', STAGE3_DIR)
print('   config:', S3['spec']['config'])

if EVAL_DIR is None:
    options = [Path('eval'), Path('etapa2_congelada/eval'), DOCUMENTS_PATH.parent / 'etapa2_congelada/eval',
               DOCUMENTS_PATH.parent / 'eval_review_v2'] + sorted(p.parent for p in DOCUMENTS_PATH.parent.glob('*/FROZEN.json'))
    matching = [p for p in dict.fromkeys(o.resolve() for o in options if (o / 'FROZEN.json').is_file())
                if json.loads((p / 'FROZEN.json').read_text(encoding='utf-8')) == S3['spec']['eval']]
    if not matching:
        raise FileNotFoundError('No encontré la evaluación congelada que usó la Etapa 3. Fijá EVAL_DIR.')
    EVAL_DIR = matching[0]
EVAL_DIR = Path(EVAL_DIR)
FROZEN = json.loads((EVAL_DIR / 'FROZEN.json').read_text(encoding='utf-8'))
if FROZEN != S3['spec']['eval']:
    raise ValueError('EVAL_DIR no es la evaluación congelada contra la que corrió la Etapa 3.')
EV = schema.load_eval(EVAL_DIR, documents_path=DOCUMENTS_PATH)       # hashes de los 3 archivos + corpus
schema.validate_eval(EVAL_DIR, DOCUMENTS_PATH).show().raise_if_failed()
if FROZEN['documents_sha256'] != S3['spec']['corpus_hash']:
    raise ValueError('El corpus de la Etapa 3 no es el de la evaluación congelada.')

ALL_ITEMS = {q['item_id']: q for q in read_jsonl(EVAL_DIR / 'single_turn.jsonl')}
ITEM_IDS = list(S3['spec']['item_ids'])
ITEMS = {i: ALL_ITEMS[i] for i in ITEM_IDS}
if any(q['split'] != EVAL_SPLIT for q in ITEMS.values()):
    raise ValueError('La corrida de la Etapa 3 tiene ítems de otro split.')
DOCS = {d['doc_id']: d for d in read_jsonl(DOCUMENTS_PATH)}
CHUNKS_BY_SECTION = {}
for c in S3['chunks'].values():
    CHUNKS_BY_SECTION.setdefault(c['section_id'], []).append(c)

pending = sum('pendiente' in q.get('notes', '').lower() for q in ITEMS.values())
if pending:
    warnings.warn(f'{pending}/{len(ITEMS)} preguntas declaran revisión humana pendiente: resultados provisionales.')
print('Evaluación:', EVAL_DIR, '| ev.key():', EV.key()[:16])
print('Perfil:', PROFILE, '| split:', EVAL_SPLIT, '| ítems:', len(ITEMS),
      '| categorías:', {c: sum(q['category'] == c for q in ITEMS.values()) for c in sorted({q['category'] for q in ITEMS.values()})})

### Guarda: cada gold tiene que existir antes de medir

Cada sección gold tiene que estar en el corpus (para el oráculo) y en el índice de la Etapa 3
(para el "recuperó bien"). Un id gold faltante, a propósito, tiene que detener la etapa.

In [ ]:
def check_gold(items, docs, chunks_by_section):
    missing = []
    for q in items:
        for g in q.get('gold') or []:
            sections = {s['section_id'] for s in docs.get(g['doc_id'], {}).get('sections', [])}
            if g.get('section_id') not in sections or g.get('section_id') not in chunks_by_section:
                missing.append((q['item_id'], g))
    if missing:
        raise ValueError(f'Gold ausente del corpus o del índice: {missing[:5]}')

check_gold(ITEMS.values(), DOCS, CHUNKS_BY_SECTION)
try:
    check_gold([{'item_id': 'guarda', 'gold': [{'doc_id': '__falta__', 'section_id': '__falta__#ART-1'}]}],
               DOCS, CHUNKS_BY_SECTION)
except ValueError as e:
    print('OK: un gold faltante se rechaza ->', clip(str(e), 120))
else:
    raise AssertionError('No saltó la guarda de gold.')

## 4. El generador: servidor, sondas y caché

`llm.setup` instala (en Colab) la versión fijada de Ollama, baja el modelo, comprueba que respete
`think=False`, mide en qué orden escribe las claves (`probe_key_order`) y calibra la cuenta de la
plantilla contra `prompt_eval_count`. El schema tiene el razonamiento primero tanto en orden de
declaración como alfabético, así que cualquiera de los dos órdenes sirve.

In [ ]:
environment.gpu_orphans()
INFO = llm.setup(MODEL, num_ctx=CFG.num_ctx)
order = INFO.get('key_order', {}).get('order')
print(verdict(order in ('declaration', 'alphabetical'),
              f'el servidor escribe las claves en orden {order}: analysis sale primero igual',
              f'orden de claves {order!r}: mirar la respuesta cruda antes de seguir'))
print(llm.placement(MODEL))
llm.use_cache(ART / 'llm_cache' / f"{MODEL.replace(':', '_').replace('/', '_')}.jsonl")


def count(text):
    return llm.count_tokens(text, MODEL)


def count_messages(messages):
    return llm.count_prompt_tokens(messages, MODEL)


if count('hola') is None:
    raise RuntimeError(f'{MODEL} no tiene tokenizer en llm.HF_TOKENIZERS: las guardas estarían apagadas.')

## 5. El ensamblador: armar los tres prompts de cada pregunta antes de llamar al modelo

- **Libro cerrado:** sin pasajes, con un prompt de sistema propio (responder con lo que sabe o abstenerse).
  Con el prompt fundamentado se abstendría siempre y la fila no mediría nada.
- **Etapa 3:** los candidatos de la Etapa 3 en su orden, hasta 2.500 tokens. Un pasaje que no entra se
  saltea y se sigue con el siguiente.
- **Oráculo:** las secciones gold enteras (todas las del conjunto gold, en su orden). Si una sección sola
  supera el presupuesto, se entregan sus hijos. En las preguntas fuera del corpus el oráculo no recibe
  ningún pasaje, porque no hay evidencia correcta que darle.

Cada prompt se cuenta completo con la plantilla de chat y se exige que entre en `num_ctx − num_predict − 64`.
Los tokens entregados por fila se reportan: el oráculo suele entregar bastante menos que la Etapa 3.

In [ ]:
RETRIEVAL_ARM = f"stage3_{S3['spec']['config'].get('name', 'retrieval')}"
ARMS = ['closed_book', RETRIEVAL_ARM, 'oracle']
PLAN = {arm: {} for arm in ARMS}
ORACLE_MODE = {}
same_as_stage3 = 0
for item_id in ITEM_IDS:
    q = ITEMS[item_id]
    PLAN['closed_book'][item_id] = fit_prompt(SYSTEM_CLOSED_BOOK, q['question'], None, CFG, count, count_messages)
    s3row = S3['rows'][item_id]
    candidates = [S3['chunks'][c['chunk_id']] for c in s3row['candidates']]
    fp = fit_prompt(SYSTEM_GROUNDED, q['question'], candidates, CFG, count, count_messages)
    PLAN[RETRIEVAL_ARM][item_id] = fp
    same_as_stage3 += [p['chunk_id'] for p in fp.delivered] == [p['chunk_id'] for p in s3row['delivered']]
    oracle_candidates, ORACLE_MODE[item_id] = oracle_passages(q, DOCS, CHUNKS_BY_SECTION, CFG.passage_budget, count)
    PLAN['oracle'][item_id] = fit_prompt(SYSTEM_GROUNDED, q['question'], oracle_candidates, CFG, count, count_messages)

for arm in ARMS:
    for item_id, fp in PLAN[arm].items():
        if not fp.fits:
            raise AssertionError(f'{arm}/{item_id}: {fp.expected_prompt_tokens} > {fp.prompt_limit}')
        if fp.passage_tokens > CFG.passage_budget:
            raise AssertionError(f'{arm}/{item_id}: pasajes {fp.passage_tokens} > {CFG.passage_budget}')

print_table('Prompts ensamblados (antes de llamar al modelo)',
            ['pasajes', 'tokens pasajes (media)', 'tokens pasajes (máx)', 'prompt (media)', 'prompt (máx)', 'salteados (media)'],
            [(arm, {'pasajes': sum(len(fp.delivered) for fp in PLAN[arm].values()) / len(ITEM_IDS),
                    'tokens pasajes (media)': sum(fp.passage_tokens for fp in PLAN[arm].values()) / len(ITEM_IDS),
                    'tokens pasajes (máx)': max(fp.passage_tokens for fp in PLAN[arm].values()),
                    'prompt (media)': sum(fp.expected_prompt_tokens for fp in PLAN[arm].values()) / len(ITEM_IDS),
                    'prompt (máx)': max(fp.expected_prompt_tokens for fp in PLAN[arm].values()),
                    'salteados (media)': sum(len(fp.skipped_ids) for fp in PLAN[arm].values()) / len(ITEM_IDS)})
             for arm in ARMS], smoke=SMOKE_TEST)
print(f'Entrega de la Etapa 3 reproducida exactamente en {same_as_stage3}/{len(ITEM_IDS)} ítems '
      '(difiere solo porque el marcador ya no lleva chunk_id: entra algún pasaje más).')
print('Oráculo con hijos en lugar de la sección entera:', [i for i, m in ORACLE_MODE.items() if m != 'section'] or 'ninguno')

# Un prompt completo, para leerlo una vez:
example = next(i for i in ITEM_IDS if ITEMS[i]['answerable'])
for m in PLAN[RETRIEVAL_ARM][example].messages:
    print(f"\n----- {m['role']} -----\n{m['content'][:2500]}" + (' […]' if len(m['content']) > 2500 else ''))

### El ensamblador saltea y sigue (no corta)

Un pasaje demasiado grande seguido de uno chico: el grande se saltea y el chico entra.

In [ ]:
_small = {'chunk_id': 'g_small', 'doc_id': 'X', 'section_id': 'X#1', 'text': 'dato breve', 'index_text': 'dato breve'}
_large = {**_small, 'chunk_id': 'g_large', 'text': 'dato largo ' * 300, 'index_text': 'dato largo ' * 300}
_kept, _skipped, _used = assemble([_large, _small], count(passage_block(_small)) + 5, count)
assert [p['chunk_id'] for p in _kept] == ['g_small'] and [p['chunk_id'] for p in _skipped] == ['g_large']
print('OK: se salteó el pasaje grande y entró el siguiente;', _used, 'tokens usados.')

## 6. Generar las tres filas

Cada llamada registra nuestra cuenta (`expected_prompt_tokens`) y la del servidor (`prompt_tokens`).
Una llamada con `r.error` (respuesta que no parsea, prompt truncado, rechazo del servidor) o un
`llm.ContextOverflow` es su propia categoría: nunca abstención, nunca respuesta puntuada.
Las filas se agregan a medida que se calculan (`ResultsLog`): si Colab se desconecta, volver a correr
retoma donde quedó, y la caché del LLM repite lo ya generado sin volver a llamar.

In [ ]:
PROMPTS_SHA256 = canonical_hash({'grounded': SYSTEM_GROUNDED, 'closed_book': SYSTEM_CLOSED_BOOK,
                                 'schema': GroundedAnswer.model_json_schema()})
RUN_SPEC = {
    'stage': '04_grounded_answers', 'profile': PROFILE, 'split': EVAL_SPLIT,
    'config': asdict(CFG), 'eval': EV.key(), 'corpus_hash': FROZEN['documents_sha256'],
    'stage3_run_key': canonical_hash(S3['spec']), 'stage3_index_key': S3['spec']['index_key'],
    'model': MODEL, 'model_digest': INFO.get('digest'), 'ollama_server': INFO.get('server'),
    'prompts_sha256': PROMPTS_SHA256, 'implementation_sha256': IMPLEMENTATION_SHA256,
    'arms': ARMS, 'item_ids': ITEM_IDS,
}
RUN_KEY = canonical_hash(RUN_SPEC)
RESULTS_DIR = ART / 'results' / f'stage4_{RUN_KEY[:16]}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
write_json(RESULTS_DIR / 'run_spec.json', RUN_SPEC)
LOG = ResultsLog(RESULTS_DIR / 'answers.jsonl', RUN_SPEC)
run = StageRun.start('04_grounded_answers', ART, profile=PROFILE, config=asdict(CFG))
started_at = time.strftime('%Y-%m-%d %H:%M:%S')

todo = [(arm, i) for arm in ARMS for i in ITEM_IDS if not LOG.done(arm, i)]
print(f'Llamadas pendientes: {len(todo)} de {len(ARMS) * len(ITEM_IDS)} '
      f'(~{len(todo) * 15 / 60:.0f} min a ~15 s por respuesta en una T4, menos si están en caché).')


def call_model(messages, **kw):
    """(respuesta o None, ChatResult o None, error). ContextOverflow se atrapa y se registra."""
    try:
        answer, r = llm.structured(messages, GroundedAnswer, model=MODEL, num_ctx=CFG.num_ctx,
                                   num_predict=CFG.num_predict, temperature=CFG.temperature,
                                   seed=CFG.seed, think=CFG.think, **kw)
    except llm.ContextOverflow as e:
        return None, None, f'CONTEXT_OVERFLOW (client): {e}'
    return answer, r, r.error


def result_row(arm, item_id, fp, answer, r, error, extra=None):
    q = ITEMS[item_id]
    metrics = score_answer(q, answer, error, fp.delivered if hasattr(fp, 'delivered') else fp,
                           closed_book=(arm == 'closed_book'))
    row = {
        'stage': '04_grounded_answers', 'arm': arm, 'item_id': item_id, 'category': q['category'],
        'question': q['question'], 'answer_key': q.get('answer_key'), 'gold': q.get('gold'),
        'metrics': metrics, 'error': error,
        'llm': r.log() if r is not None else None,
        'output': answer.model_dump() if answer is not None else None,
        'raw_text': (r.text[:2000] if (r is not None and answer is None) else None),
        'delivered': [{k: p[k] for k in ('chunk_id', 'doc_id', 'section_id')} for p in
                      (fp.delivered if hasattr(fp, 'delivered') else fp)],
        'skipped_ids': getattr(fp, 'skipped_ids', []),
        'passage_tokens': getattr(fp, 'passage_tokens', None),
        'assembler_expected_prompt_tokens': getattr(fp, 'expected_prompt_tokens', None),
    }
    return {**row, **(extra or {})}


t_start = time.time()
for n, (arm, item_id) in enumerate(todo, 1):
    fp = PLAN[arm][item_id]
    answer, r, error = call_model(fp.messages)
    row = result_row(arm, item_id, fp, answer, r, error)
    LOG.write(row)
    m = row['metrics']
    tokens = '' if r is None else f'prompt {r.expected_prompt_tokens}/{r.prompt_tokens} (nuestra/servidor)'
    print(f"[{n}/{len(todo)}] {arm:<14} {item_id} {m['outcome']:<9} correct={m['correct']} {tokens}"
          + (f' ERROR {clip(error, 90)}' if error else ''))
print(f'Generación: {time.time() - t_start:.0f} s')

ROWS = {arm: sorted([r for r in LOG.rows(arm) if r['item_id'] in ITEMS], key=lambda r: ITEM_IDS.index(r['item_id']))
        for arm in ARMS}
assert all(len(ROWS[a]) == len(ITEM_IDS) for a in ARMS), 'faltan filas: volvé a correr la celda'

## 7. Las tres filas

- `correct`: con respuesta → respondió (`can_answer`) + clave contenida (+ veredicto sí/no si corresponde);
  fuera del corpus → se abstuvo. Los errores no se puntúan y se cuentan aparte.
- `cited_not_delivered`: citó un `section_id` que no se le entregó. `cited_gold`: citó una sección gold.
- `abst_con_resp`: se abstuvo con la sección gold o la clave en su contexto.
- `answer_delivered` es contención léxica en los pasajes entregados (recuperación), no corrección.
- IC 95%: bootstrap sobre preguntas. Con estas n, los intervalos son anchos: leelos antes que las medias.

In [ ]:
SUMMARY = {arm: summarize_arm(ROWS[arm], CFG) for arm in ARMS}
cols = ['n', 'errors', 'correct', 'correct_ci95_low', 'correct_ci95_high', 'correct_answerable',
        'abstention_ooc', 'key_hit', 'verdict_ok', 'cited_not_delivered', 'cited_gold',
        'abstained_with_answer_in_context', 'mean_passage_tokens']
print_table('Etapa 4 · tres filas', cols, [(a, SUMMARY[a]) for a in ARMS], smoke=SMOKE_TEST)
print_table('Recuperación entregada y cuentas de tokens', ['gold_delivered', 'answer_delivered',
            'mean_expected_prompt_tokens', 'mean_server_prompt_tokens', 'max_abs_delta_tokens', 'mean_ms', 'flags'],
            [(a, SUMMARY[a]) for a in ARMS], smoke=SMOKE_TEST)

BY_CATEGORY = {}
for arm in ARMS:
    for cat in sorted({q['category'] for q in ITEMS.values()}):
        rows = [r for r in ROWS[arm] if r['category'] == cat and r['metrics']['outcome'] != ERROR]
        BY_CATEGORY.setdefault(cat, {})[arm] = (sum(bool(r['metrics']['correct']) for r in rows) / len(rows)) if rows else None
print_table('correct por categoría', ARMS, [(c, v) for c, v in BY_CATEGORY.items()], smoke=SMOKE_TEST)

cb = SUMMARY['closed_book']['correct_answerable']
if cb is not None and cb >= 0.5:
    print('CHECK -- a libro cerrado se responde la mitad o más: el conjunto evalúa preentrenamiento, no recuperación.')
deltas = [r['llm']['expected_prompt_tokens'] - r['llm']['prompt_tokens'] for a in ARMS for r in ROWS[a]
          if r.get('llm') and r['llm'].get('prompt_tokens') is not None and r['llm'].get('expected_prompt_tokens') is not None]
if deltas:
    print(verdict(max(abs(d) for d in deltas) <= 16,
                  f'nuestra cuenta y prompt_eval_count coinciden (|Δ| máx {max(abs(d) for d in deltas)}, n={len(deltas)})',
                  f'|Δ| máx {max(abs(d) for d in deltas)} tokens: revisar (un Δ positivo grande es truncamiento)'))
errors = [(a, r['item_id'], r['error']) for a in ARMS for r in ROWS[a] if r['metrics']['outcome'] == ERROR]
for a, i, e in errors:
    print('ERROR', a, i, clip(e, 140))

## 8. La tabla 2×2 y abrir la caja

Por pregunta con respuesta: **recuperó bien / mal** (la sección gold llegó al prompt) × **respondió bien / mal**.

- *Fiel al pasaje equivocado:* recuperó mal y respondió lo que decía otro pasaje (citó un pasaje entregado
  que no es gold). Es un candidato: confirmalo leyendo la respuesta.
- *Lo tenía, respondió otra cosa:* recuperó bien y respondió mal.

Se repite con "recuperó bien" = la clave está en el texto entregado, porque una sección gold larga puede
llegar como un hijo que no contiene la respuesta.

In [ ]:
TT = two_by_two(ROWS[RETRIEVAL_ARM], 'gold_delivered')
TT_KEY = two_by_two(ROWS[RETRIEVAL_ARM], 'answer_delivered')
for tt in (TT, TT_KEY):
    c = tt['counts']
    print(f"\n2×2 · {RETRIEVAL_ARM} · recuperó bien = {tt['retrieval_criterion']}"
          + (f"   [{'SMOKE'}]" if SMOKE_TEST else ''))
    print(f"{'':22}{'respondió bien':>16}{'respondió mal':>16}")
    print(f"{'recuperó bien':22}{c['ret_ok_ans_ok']:>16}{c['ret_ok_ans_bad']:>16}")
    print(f"{'recuperó mal':22}{c['ret_bad_ans_ok']:>16}{c['ret_bad_ans_bad']:>16}")
    print(f"errores (aparte): {c['error']}")
    got = c['ret_ok_ans_ok'] + c['ret_ok_ans_bad']
    if got:
        print(f"De las {got} preguntas en las que llegó, {c['ret_ok_ans_ok']} se respondieron bien.")

FAILURES = []
for arm in ARMS:
    for r in ROWS[arm]:
        kind = failure_kind(r['metrics'])
        if kind == 'ok':
            continue
        out = r.get('output') or {}
        FAILURES.append({'arm': arm, 'item_id': r['item_id'], 'kind': kind, 'category': r['category'],
                         'question': r['question'], 'answer_key': r['answer_key'],
                         'gold': [g.get('section_id') for g in r['gold'] or []],
                         'delivered_sections': sorted({p['section_id'] for p in r['delivered']}),
                         'answer': out.get('answer'), 'analysis': out.get('analysis'),
                         'citations': out.get('citations'), 'verdict': out.get('verdict'), 'error': r['error']})
for f in FAILURES:
    if f['arm'] == 'closed_book':
        continue
    print(f"\n[{f['arm']}] {f['item_id']} · {f['kind']}\n   P: {clip(f['question'], 150)}")
    print(f"   clave: {f['answer_key']} | gold: {f['gold']}")
    print(f"   entregadas: {clip(', '.join(f['delivered_sections']), 150)}")
    print(f"   respuesta: {clip(f['answer'] or f['error'], 220)}")
    print(f"   análisis: {clip(f['analysis'] or '', 220)}")
    print(f"   citas: {f['citations']} · veredicto: {f['verdict']}")

FLAGS = [(a, r['item_id'], [k for k in ('flag_says_cannot_but_answers', 'flag_says_can_but_abstains',
                                         'flag_answers_without_citation') if r['metrics'].get(k)])
         for a in ARMS for r in ROWS[a] if r['metrics']['outcome'] != ERROR]
FLAGS = [x for x in FLAGS if x[2]]
print('\nFlags de consistencia (heurísticos, para leer a mano):', FLAGS or 'ninguno')

## 9. Brechas, citas y abstenciones, con su cota de ruido

Comparaciones pareadas sobre las mismas preguntas: solo las discordantes llevan información.
Test de signos bilateral exacto y bootstrap de la diferencia. Los ítems con error en cualquiera
de las dos filas se excluyen y se cuentan.

In [ ]:
PAIRED = {
    f'oracle vs {RETRIEVAL_ARM}': paired_comparison(ROWS[RETRIEVAL_ARM], ROWS['oracle']),
    f'{RETRIEVAL_ARM} vs closed_book': paired_comparison(ROWS['closed_book'], ROWS[RETRIEVAL_ARM]),
    'oracle vs closed_book': paired_comparison(ROWS['closed_book'], ROWS['oracle']),
}
print_table('Comparaciones pareadas (B vs A: victorias de B)', ['wins', 'losses', 'ties', 'excluded_errors',
            'delta', 'sign_test_p_two_sided'], list(PAIRED.items()), smoke=SMOKE_TEST)
for k, v in PAIRED.items():
    print(f"{k}: Δ IC95 {v['delta_ci95']}")
need = min_discordant_for_significance()
print(f'Con {len(ITEM_IDS)} preguntas, una diferencia es distinguible del azar (p<0,05) solo si hay al menos '
      f'{need} discordantes y todas en el mismo sentido.')

for arm in [RETRIEVAL_ARM, 'oracle', 'closed_book']:
    bad_cite = [r['item_id'] for r in ROWS[arm] if r['metrics'].get('cited_not_delivered')]
    abst_ctx = [r['item_id'] for r in ROWS[arm] if r['metrics'].get('abstained_with_answer_in_context')]
    print(f"{arm}: citó algo no entregado en {len(bad_cite)} {bad_cite} · "
          f"se abstuvo con la respuesta en el contexto en {len(abst_ctx)} {abst_ctx}")

## 10. Ver saltar la guarda, a propósito

Un prompt deliberadamente más grande que `num_ctx`, con la sección gold PRIMERO (el pasaje mejor rankeado):

1. Con `strict_ctx=True` (lo normal): `llm.chat` cuenta el prompt y lanza `ContextOverflow` sin mandar nada.
2. Con `strict_ctx=False`, una sola vez: el servidor conserva unos pocos tokens del frente y la cola, y
   descarta el medio: las instrucciones, la regla de abstención y los mejores pasajes. La pregunta del
   final sobrevive, el modelo responde con fluidez y `format` hace que la salida siga siendo JSON válido.
   `llm.chat` lo detecta comparando su cuenta con `prompt_eval_count` (`SILENT_TRUNCATION`).

Mismo `num_ctx` que el resto (no recarga el modelo) y sin caché.

In [ ]:
GUARD = None
if RUN_GUARD_DEMO:
    gid = next(i for i in ITEM_IDS if ITEMS[i]['answerable'])
    q = ITEMS[gid]
    oracle_first, _ = oracle_passages(q, DOCS, CHUNKS_BY_SECTION, 10 ** 9, count)
    pool = oracle_first + [S3['chunks'][c['chunk_id']] for c in S3['rows'][gid]['candidates']]
    pool = pool + sorted(S3['chunks'].values(), key=lambda c: c['chunk_id'])   # relleno si faltan candidatos
    big, _, big_tokens = assemble(pool, int(CFG.num_ctx * 1.4), count)
    big_messages = build_messages(SYSTEM_GROUNDED, q['question'], big)
    ours = count_messages(big_messages)
    print(f'Prompt de prueba: {len(big)} pasajes, {ours} tokens con plantilla; num_ctx {CFG.num_ctx}, '
          f'num_predict {CFG.num_predict}.')
    try:
        llm.structured(big_messages, GroundedAnswer, model=MODEL, num_ctx=CFG.num_ctx, num_predict=CFG.num_predict,
                       temperature=0, seed=CFG.seed, think=False, cache=False)
    except llm.ContextOverflow as e:
        strict_outcome = f'ContextOverflow: {e}'
        print('1) strict_ctx=True ->', strict_outcome)
    else:
        raise AssertionError('La guarda no saltó con un prompt más grande que num_ctx.')
    ans_big, r_big = llm.structured(big_messages, GroundedAnswer, model=MODEL, num_ctx=CFG.num_ctx,
                                    num_predict=CFG.num_predict, temperature=0, seed=CFG.seed, think=False,
                                    cache=False, strict_ctx=False)
    GUARD = {'item_id': gid, 'question': q['question'], 'answer_key': q.get('answer_key'),
             'passages': len(big), 'our_count': ours, 'num_ctx': CFG.num_ctx, 'num_predict': CFG.num_predict,
             'strict_outcome': strict_outcome, 'server_prompt_eval_count': r_big.prompt_tokens,
             'done_reason': r_big.done_reason, 'error': r_big.error, 'raw_reply': r_big.text[:3000],
             'gold_first_section': oracle_first[0]['section_id'] if oracle_first else None}
    print(f"2) strict_ctx=False -> nuestra cuenta {ours}, el servidor evaluó {r_big.prompt_tokens}; "
          f"error={r_big.error!r}; done_reason={r_big.done_reason}")
    print('   respuesta del modelo (cruda):')
    print(wrap(r_big.text, max_lines=12))
    print(f"   ¿contiene la clave? {contains_key(r_big.text, q.get('answer_key'))}")
    write_json(RESULTS_DIR / 'guard_demo.json', GUARD)

## 11. Opcional: posición del pasaje con la respuesta

Con `RUN_POSITION = True`. Para cada pregunta con respuesta: el hijo gold que contiene la clave más
`POSITION_N_PASSAGES − 1` distractores (candidatos de la Etapa 3 de secciones no gold y sin la clave).
El mismo conjunto de pasajes en tres órdenes: la respuesta primero, en el medio o última. Igual
cantidad de pasajes y de tokens en las tres. Con ~12 pasajes en ~5.000 tokens la ventana es chica
para *Lost in the Middle*: un nulo acá no lo refuta. Costo: 3 llamadas por pregunta.

In [ ]:
POSITION = None
if RUN_POSITION:
    POS_LOG = ResultsLog(RESULTS_DIR / 'position.jsonl', {**RUN_SPEC, 'position': {
        'n_passages': POSITION_N_PASSAGES, 'budget': POSITION_BUDGET}})
    pos_plan, excluded = {}, {}
    for item_id in ITEM_IDS:
        q = ITEMS[item_id]
        if not q['answerable'] or not q.get('answer_key'):
            continue
        gold_secs = {g['section_id'] for g in q['gold']}
        needle = next((c for s in [g['section_id'] for g in q['gold']]
                       for c in sorted(CHUNKS_BY_SECTION.get(s, []), key=lambda c: c['start'])
                       if contains_key(c['text'], q['answer_key'])), None)
        if needle is None:
            excluded[item_id] = 'ningún hijo gold contiene la clave'
            continue
        distractors = []
        for c in S3['rows'][item_id]['candidates']:
            ch = S3['chunks'][c['chunk_id']]
            if ch['section_id'] in gold_secs or contains_key(ch['text'], q['answer_key']):
                continue
            if count(render_passages(distractors + [ch, needle])) <= POSITION_BUDGET:
                distractors.append(ch)
            if len(distractors) == POSITION_N_PASSAGES - 1:
                break
        if len(distractors) < POSITION_N_PASSAGES - 1:
            excluded[item_id] = f'solo {len(distractors)} distractores entran'
            continue
        k = len(distractors)
        orders = {'first': [needle] + distractors,
                  'middle': distractors[:k // 2] + [needle] + distractors[k // 2:],
                  'last': distractors + [needle]}
        sizes = {pos: count_messages(build_messages(SYSTEM_GROUNDED, q['question'], ps)) for pos, ps in orders.items()}
        if max(sizes.values()) - min(sizes.values()) > 4 or max(sizes.values()) > CFG.prompt_limit:
            excluded[item_id] = f'tokens desiguales o excedidos {sizes}'
            continue
        pos_plan[item_id] = (orders, sizes)
    print(f'Prueba de posición: {len(pos_plan)} preguntas; excluidas: {excluded or "ninguna"}')
    for item_id, (orders, sizes) in pos_plan.items():
        for pos, ps in orders.items():
            arm = f'pos_{pos}'
            if POS_LOG.done(arm, item_id):
                continue
            answer, r, error = call_model(build_messages(SYSTEM_GROUNDED, ITEMS[item_id]['question'], ps))
            POS_LOG.write(result_row(arm, item_id, ps, answer, r, error,
                                     {'position': pos, 'prompt_tokens_by_position': sizes}))
    pos_rows = {pos: sorted([r for r in POS_LOG.rows(f'pos_{pos}') if r['item_id'] in pos_plan],
                            key=lambda r: r['item_id']) for pos in ('first', 'middle', 'last')}
    POSITION = {'n_items': len(pos_plan), 'excluded': excluded,
                'correct': {p: summarize_arm(rs, CFG)['correct'] for p, rs in pos_rows.items() if rs},
                'paired': {'first vs middle': paired_comparison(pos_rows['middle'], pos_rows['first']),
                           'last vs middle': paired_comparison(pos_rows['middle'], pos_rows['last'])}
                if pos_plan else {}}
    print(json.dumps(POSITION, indent=2, ensure_ascii=False))
    print(f'Con {len(pos_plan)} preguntas, un efecto de posición solo sería significativo (p<0,05) con ≥ {need} '
          'discordantes en el mismo sentido' + (': con estas preguntas es imposible, y un nulo no dice nada.'
                                                if len(pos_plan) < need else '.'))
    write_json(RESULTS_DIR / 'position_summary.json', POSITION)

## 12. Reproducibilidad: repetir unas llamadas sin caché

A temperatura 0 y semilla fija, la misma llamada debería dar la misma salida. Si varía, esa varianza
es parte del piso de ruido (§7.4).

In [ ]:
REPRO = []
for item_id in ITEM_IDS[:REPRO_CHECK_N]:
    fp = PLAN[RETRIEVAL_ARM][item_id]
    answer, r, error = call_model(fp.messages, cache=False)
    before = next(x for x in ROWS[RETRIEVAL_ARM] if x['item_id'] == item_id).get('output')
    REPRO.append({'item_id': item_id, 'identical': (answer.model_dump() if answer else None) == before,
                  'error': error})
print(verdict(all(x['identical'] for x in REPRO), f'{len(REPRO)}/{len(REPRO)} salidas idénticas al repetir',
              f"{sum(x['identical'] for x in REPRO)}/{len(REPRO)} idénticas: hay varianza entre corridas"))

## 13. Guardar tablas e informe de la corrida

`stage4_report_tables.md` trae las tablas y los números de las preguntas del informe ya calculados.
La interpretación queda para que la escriban ustedes.

In [ ]:
with (RESULTS_DIR / 'summary.csv').open('w', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['arm'] + list(SUMMARY[ARMS[0]]))
    w.writeheader()
    for a in ARMS:
        w.writerow({'arm': a, **SUMMARY[a]})
write_json(RESULTS_DIR / 'summary.json', {'summary': SUMMARY, 'by_category': BY_CATEGORY})
write_json(RESULTS_DIR / 'two_by_two.json', {'gold_delivered': TT, 'answer_delivered': TT_KEY})
write_json(RESULTS_DIR / 'paired.json', PAIRED)
write_jsonl(RESULTS_DIR / 'failures.jsonl', FAILURES)
write_json(RESULTS_DIR / 'repro_check.json', REPRO)


def _f(x, pct=True):
    return '—' if x is None else (f'{100 * x:.0f}%' if pct else f'{x:,.0f}')


md = [f'# Etapa 4 · tablas ({PROFILE}, split {EVAL_SPLIT}, n={len(ITEM_IDS)})', '',
      f'Modelo `{MODEL}` (digest {INFO.get("digest")}), num_ctx {CFG.num_ctx}, num_predict {CFG.num_predict}, '
      f'pasajes ≤ {CFG.passage_budget} tokens. Corrida `{RUN_KEY[:16]}`; Etapa 3 `{STAGE3_DIR.name}`.', '']
if SMOKE_TEST:
    md += ['> SMOKE_TEST: estos números no significan nada; solo prueban que el código corre.', '']
md += ['## Tres filas', '',
       '| fila | correct (IC95) | con respuesta | abstención OOC | clave | veredicto | citó no entregado | citó gold | abst. con resp. en contexto | tokens pasajes | errores |',
       '|---|---|---|---|---|---|---|---|---|---|---|']
for a in ARMS:
    s = SUMMARY[a]
    md.append(f"| {a} | {_f(s['correct'])} ({_f(s['correct_ci95_low'])}–{_f(s['correct_ci95_high'])}) | "
              f"{_f(s['correct_answerable'])} | {_f(s['abstention_ooc'])} | {_f(s['key_hit'])} | {_f(s['verdict_ok'])} | "
              f"{_f(s['cited_not_delivered'])} | {_f(s['cited_gold'])} | {_f(s['abstained_with_answer_in_context'])} | "
              f"{_f(s['mean_passage_tokens'], False)} | {s['errors']} |")
c = TT['counts']
md += ['', f'## 2×2 · {RETRIEVAL_ARM} (recuperó bien = sección gold entregada)', '',
       '| | respondió bien | respondió mal |', '|---|---|---|',
       f"| recuperó bien | {c['ret_ok_ans_ok']} | {c['ret_ok_ans_bad']} |",
       f"| recuperó mal | {c['ret_bad_ans_ok']} | {c['ret_bad_ans_bad']} |", '',
       f"Errores aparte: {c['error']}. Con \"recuperó bien\" = clave en el texto entregado: {TT_KEY['counts']}.", '',
       '## Comparaciones pareadas (correct)', '', '| comparación | victorias | derrotas | empates | Δ | IC95 Δ | p (signos) |',
       '|---|---|---|---|---|---|---|']
for k, v in PAIRED.items():
    lo, hi = v['delta_ci95']
    md.append(f"| {k} | {v['wins']} | {v['losses']} | {v['ties']} | {_f(v['delta'])} | {_f(lo)}–{_f(hi)} | "
              f"{v['sign_test_p_two_sided']:.3f} |")
md += ['', '## Números para las preguntas del informe', '',
       f"- Llegó la sección gold en {c['ret_ok_ans_ok'] + c['ret_ok_ans_bad']} preguntas; se respondieron bien "
       f"{c['ret_ok_ans_ok']}. Fallas: " + ', '.join(sorted({f['kind'] for f in FAILURES if f['arm'] == RETRIEVAL_ARM})) + '.',
       f"- Brecha oráculo − {RETRIEVAL_ARM}: {_f(PAIRED[f'oracle vs {RETRIEVAL_ARM}']['delta'])}; "
       f"{RETRIEVAL_ARM} − libro cerrado: {_f(PAIRED[f'{RETRIEVAL_ARM} vs closed_book']['delta'])}.",
       '- Citó un pasaje no entregado: ' + '; '.join(f"{a} {_f(SUMMARY[a]['cited_not_delivered'])}" for a in ARMS) + '.',
       '- Se abstuvo con la respuesta en el contexto: ' + '; '.join(
           f"{a} {_f(SUMMARY[a]['abstained_with_answer_in_context'])}" for a in (RETRIEVAL_ARM, 'oracle')) + '.']
if GUARD:
    md.append(f"- Prompt demasiado grande: nuestra cuenta {GUARD['our_count']}, el servidor evaluó "
              f"{GUARD['server_prompt_eval_count']} (num_ctx {GUARD['num_ctx']}); error `{GUARD['error']}`. "
              f"Respuesta: «{clip(GUARD['raw_reply'], 300)}»")
if POSITION:
    md.append('- Posición: correct ' + ', '.join(f'{p} {_f(v)}' for p, v in POSITION['correct'].items())
              + f" sobre {POSITION['n_items']} preguntas (pareadas: {', '.join(f'{k} p={v['sign_test_p_two_sided']:.2f}' for k, v in POSITION['paired'].items())}).")
md += ['', '## Interpretación', '', '(Escribila ustedes: qué leen en estos números y qué harían distinto.)', '']
(RESULTS_DIR / 'stage4_report_tables.md').write_text('\n'.join(md), encoding='utf-8')

report = {
    'stage': '04_grounded_answers', 'status': 'ok', 'profile': PROFILE, 'split': EVAL_SPLIT,
    'started_at': started_at, 'finished_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'run_key': RUN_KEY, 'run_spec': RUN_SPEC, 'results_dir': str(RESULTS_DIR), 'stage3_dir': str(STAGE3_DIR),
    'llm_setup': INFO, 'llm_stats': dict(llm.STATS), 'items': len(ITEM_IDS),
    'summary': SUMMARY, 'two_by_two': TT['counts'], 'paired': PAIRED,
    'guard_demo': GUARD, 'position': POSITION, 'repro_check': REPRO,
    'oracle_children_fallback': [i for i, m in ORACLE_MODE.items() if m != 'section'],
    'human_review_pending_in_notes': pending,
    'limitations': ['smoke is not a benchmark' if SMOKE_TEST else f'n={len(ITEM_IDS)}: read the CIs',
                    'key containment is lexical; correctness is not certified',
                    'consistency flags are heuristic and only for manual inspection',
                    'oracle delivers fewer tokens than the retrieval row'],
}
write_json(RESULTS_DIR / 'run_report.json', report)
run.note('results_dir', str(RESULTS_DIR))
run.note('items', len(ITEM_IDS))
run.note('summary', {a: SUMMARY[a]['correct'] for a in ARMS})
run.finish()
print('Resultados:', RESULTS_DIR)
print('Para test: SMOKE_TEST=False, EVAL_SPLIT="test" (después de correr la Etapa 3 en test).')

## 14. Preguntas para el informe (completar después de correr en test)

- De las preguntas en las que llegó el pasaje con la respuesta, ¿cuántas se respondieron bien, y cómo eran las fallas?
- ¿Qué tan grande es la brecha entre la fila oráculo y la de la Etapa 3? ¿Y entre libro cerrado y la de la Etapa 3?
- ¿Con qué frecuencia el modelo citó un pasaje que no se le dio, o se abstuvo mientras la respuesta estaba en su contexto?
- Mostrá un prompt demasiado grande: tu cuenta, la cuenta del servidor, y qué respondió el modelo (`guard_demo.json`).
- ¿Importó la posición a tu escala, y podría tu prueba haberlo mostrado si importara?